# FQ-SSIUU Mother Notebook v2 (Local Windows)

**Local execution version** — designed to run on Windows with RTX 3060 12GB,
replacing the Kaggle container environment.

## Quick Start
1. Set `HF_TOKEN` in your system environment (see README)
2. Open this notebook in Jupyter (File → Change Kernel → `.venv/Scripts/python.exe`)
3. Run all cells sequentially
4. Outputs saved to `./runs/<run-id>/`

**Prerequisites:** Python 3.11+, CUDA-capable GPU (12GB VRAM minimum),
virtual environment with dependencies installed.

---
**Author:** Rohan Ahmed (23BSAI008), Muhammad Arslan (23BSAI020)
**Supervisor:** Dr. Sania Bhatti
**Project:** FQ-SSIUU — Verifiable Machine Unlearning Against Quantisation Recovery Attacks

In [ ]:
# ── Local Configuration (EDIT ME) ──
import os
from pathlib import Path

# 1. Project root (where this notebook lives)
ROOT = Path.cwd()

# 2. Model config
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.environ.get("HF_TOKEN", "PENDING")
if HF_TOKEN == "PENDING":
    raise RuntimeError(
        "HF_TOKEN not set in environment. "
        "Set it once: setx HF_TOKEN your-token-here (or add to .env)"
    )

# 3. Output paths (local)
RUNS_DIR = ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Recipe to execute (integration test)
RECIPE_PATH = ROOT / "configs" / "runs" / "integration_v1.json"
if not RECIPE_PATH.exists():
    raise FileNotFoundError(f"Recipe not found: {RECIPE_PATH}")

print(f"ROOT:        {ROOT}")
print(f"Recipe:      {RECIPE_PATH}")
print(f"Runs dir:    {RUNS_DIR}")
print(f"HF_TOKEN:    {'**set**' if HF_TOKEN != 'PENDING' else 'NOT SET'}")
print(f"Python:      {sys.version.split()[0]}")
print(f"CUDA:        {('Available (GPU)' if torch.cuda.is_available() else 'CPU-only (slow)')}")

In [ ]:
# ── Imports & Setup ──
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch
import numpy as np

# Verify critical imports work
for pkg in ["torch", "transformers", "peft", "datasets", "bitsandbytes"]:
    try:
        __import__(pkg)
        print(f"  OK  {pkg}")
    except ImportError as e:
        print(f"  FAIL {pkg}: {e}")

# GPU check
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")

# Working directories
run_uuid = uuid.uuid4().hex[:12]
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + run_uuid
checkout_dir = ROOT / "src-checkout" / run_id
provenance_path = RUNS_DIR / f"bootstrap-{run_id}.json"
output_dir = RUNS_DIR / run_id
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nRun ID: {run_id}")
print(f"Checkout: {checkout_dir}")
print(f"Output: {output_dir}")

In [ ]:
# ── Source Manifest & Checkout ──
import json

try:
    # 1. Load source manifest
    manifest_path = ROOT / "SOURCE_MANIFEST.json"
    if not manifest_path.exists():
        # Fallback: try to generate one from git
        print("  -> No SOURCE_MANIFEST.json found. Attempting to generate from git...")
        git_info = {
            "source_origin_sha": subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
            ).strip()[:12],
            "source_tree_sha": subprocess.check_output(
                ["git", "rev-parse", "HEAD^{tree}"], cwd=ROOT, text=True
            ).strip()[:12],
            "created_local": True,
        }
        with manifest_path.open("w") as f:
            json.dump(git_info, f, indent=2)
        print(f"  -> Generated local manifest: {manifest_path}")
    else:
        manifest = json.loads(manifest_path.read_text())
        print(f"  OK  Loaded manifest: {manifest_path}")

    # 2. Verify source tree SHA
    expected_tree = subprocess.check_output(
        ["git", "rev-parse", "HEAD^{tree}"], cwd=ROOT, text=True
    ).strip()[:12]
    print(f"  Source tree SHA: {expected_tree}")

    # 3. Clone repo to checkout dir
    if checkout_dir.exists():
        shutil.rmtree(checkout_dir)
    checkout_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--no-checkout", str(ROOT), str(checkout_dir)],
                   check=True, cwd=ROOT)
    subprocess.run(["git", "checkout", "HEAD", "--", "."], cwd=checkout_dir, check=True)

    # 4. Verify repo is clean
    status = subprocess.run(["git", "status", "--porcelain"], cwd=checkout_dir,
                            capture_output=True, text=True)
    if status.stdout.strip():
        print("  WARNING: Working tree has uncommitted changes. Proceeding...")
    else:
        print("  OK  Source checkout is clean.")

    print("  OK  Source ready for execution.")

except Exception as exc:
    print(f"  FAIL: Source setup failed: {exc}")
    raise

In [ ]:
# ── Model Loading (NF4 4-bit) ──
from fq_ssiuu.models.loader import load_base_model, ModelConfig

print("Loading base model (NF4 4-bit)...")
print(f"Model: {BASE_MODEL_ID}")

# Check VRAM headroom
if torch.cuda.is_available():
    free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"Free VRAM before load: {round(free_mem / 1024**3, 1)} GB")

config = ModelConfig(
    base_model_id=BASE_MODEL_ID,
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    torch_dtype="float16",
    device_map={"": 0},
)

model = load_base_model(config)

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    res = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory: {alloc:.2f} GB allocated, {res:.2f} GB reserved")

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("OK  Model loaded successfully.")

In [ ]:
# ── LoRA Wrapping ──
from peft import LoraConfig, get_peft_model

print("Applying LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"LoRA applied. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Check VRAM after LoRA
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    res = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory: {alloc:.2f} GB allocated, {res:.2f} GB reserved")
    if res > 8.0:
        print(f"  WARNING: Only {res:.1f} GB free - may OOM on training")

print("OK  LoRA ready.")

In [ ]:
# ── Run the Mother Runner (Integration Recipe) ──
import subprocess
import json

print("Executing mother_runner.py...")
print(f"Recipe: {RECIPE_PATH}")
print(f"Output: {output_dir}")

mother_runner = ROOT / "scripts" / "mother_runner.py"
if not mother_runner.exists():
    print("  FAIL: mother_runner.py not found. Skipping execution.")
    raise FileNotFoundError("mother_runner.py not found")

command = [
    sys.executable, str(mother_runner),
    "--recipe", str(RECIPE_PATH),
    "--output", str(output_dir),
    "--provenance", str(provenance_path),
    "--run-id", run_id,
]

env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
print(f"Running: {' '.join(command[:3])}...")

result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=3600)

print("STDOUT (tail):")
print(result.stdout[-1000:] if result.stdout else "None")
print("STDERR (tail):")
print(result.stderr[-1000:] if result.stderr else "None")
print(f"Return code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(f"Recipe failed with code {result.returncode}")

print(f"\nOK  Run complete: {run_id}")
print(f"  Outputs: {output_dir}")

In [ ]:
# ── Save Provenance & Summary ──
import json as json_mod
from datetime import UTC

provenance = {
    "run_id": run_id,
    "source_origin_sha": subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()[:12],
    "git_branch": subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only",
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0,
    "python_version": sys.version,
    "timestamp": datetime.now(UTC).isoformat(),
    "status": "SUCCESS" if result.returncode == 0 else "FAILURE",
}

with provenance_path.open("x") as f:
    json_mod.dump(provenance, f, indent=2)

print(f"OK  Provenance saved: {provenance_path}")
print(f"\n=== SUMMARY ===")
print(f"Run ID:    {run_id}")
print(f"Status:    {provenance['status']}")
print(f"GPU:       {provenance['gpu']}")
print(f"VRAM:      {provenance['vram_gb']} GB")
print(f"Outputs:   {output_dir}")

print("\n=== NEXT STEPS ===")
print("1. Inspect outputs in:  runs/<run-id>")
print("2. Run tests:            .venv/Scripts/python -m pytest tests -v")
print("3. Archive:              python scripts/archive_run.py seal --run-dir runs/<run-id> --output runs/sealed --run-id <run-id>")
print("4. Save to git:          git add -A && git commit -m 'Local execution' && git push")
print("5. View report:          .venv/Scripts/python -m markdown <output>.md")

In [ ]:
# ── Cleanup ──
import shutil

if checkout_dir.exists():
    shutil.rmtree(checkout_dir)
    print(f"OK  Cleanup: Removed {checkout_dir}")

print("\nNotebook execution complete.")
print("See 'runs/' for all run artifacts.")